In [1]:
import tensorflow as tf
import numpy as np
IMG_SIZE = (224, 224)   # 필요에 따라 (128,128) 등으로 변경
BATCH    = 32

train_dir = "rock-paper-scissors/train"
valid_dir = "rock-paper-scissors/valid"
test_dir  = "rock-paper-scissors/test"

# 디렉터리에서 이미지 분류용 데이터셋 만들기
train_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    labels="inferred",
    label_mode="int",             # [0..C-1] 정수 라벨
    image_size=IMG_SIZE,
    batch_size=BATCH,
    shuffle=True,
    seed=42,
)

valid_ds = tf.keras.utils.image_dataset_from_directory(
    valid_dir,
    labels="inferred",
    label_mode="int",
    image_size=IMG_SIZE,
    batch_size=BATCH,
    shuffle=True,
    seed=42,
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    labels="inferred",
    label_mode="int",
    image_size=IMG_SIZE,
    batch_size=BATCH,
    shuffle=False
)
# 클래스 이름 확인
class_names = train_ds.class_names
print("class_names:", class_names)    # 예: ['paper', 'rock', 'scissors']
# --- feature와 label 일부 출력 ---
for images, labels in train_ds.take(1):
    print("features shape:", images.shape)   # (B, H, W, 3)
    print("labels shape:", labels.shape)     # (B,)
    print("labels (first 10):", labels[:10].numpy())
    print("labels mapped (first 10):", [class_names[i] for i in labels[:10].numpy()])

Found 2520 files belonging to 3 classes.
Found 372 files belonging to 3 classes.
Found 33 files belonging to 3 classes.
class_names: ['paper', 'rock', 'scissors']
features shape: (32, 224, 224, 3)
labels shape: (32,)
labels (first 10): [2 1 0 2 2 1 0 1 1 1]
labels mapped (first 10): ['scissors', 'rock', 'paper', 'scissors', 'scissors', 'rock', 'paper', 'rock', 'rock', 'rock']


In [2]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.optimizers import Adam
from sklearn.tree import plot_tree
import tensorflow as tf
from tensorflow.keras.datasets import fashion_mnist
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import numpy as np
import matplotlib.pyplot as plt
import os

In [4]:
# 데이터 전처리 (0~255 → 0~1)
normalization_layer = tf.keras.layers.Rescaling(1./255)
train_ds_norm = train_ds.map(lambda x, y: (normalization_layer(x), y))
valid_ds_norm = valid_ds.map(lambda x, y: (normalization_layer(x), y))
test_ds_norm  = test_ds.map(lambda x, y: (normalization_layer(x), y))

# CNN 모델
model = tf.keras.Sequential([
    tf.keras.layers.Conv2D(32, (3, 3), activation='relu', input_shape=IMG_SIZE + (3,)),
    tf.keras.layers.MaxPooling2D(2, 2),
    tf.keras.layers.Conv2D(64, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D(2, 2),
    tf.keras.layers.Conv2D(128, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D(2, 2),
    tf.keras.layers.Flatten(),                                                                                                                                    
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(len(class_names), activation='softmax')
])
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()
history = model.fit(train_ds_norm,  validation_data=valid_ds_norm, epochs=100, batch_size=64, verbose=2)
earlystop = EarlyStopping(monitor='val_loss', patience=5)

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d_3 (Conv2D)           (None, 222, 222, 32)      896       
                                                                 
 max_pooling2d_3 (MaxPooling  (None, 111, 111, 32)     0         
 2D)                                                             
                                                                 
 conv2d_4 (Conv2D)           (None, 109, 109, 64)      18496     
                                                                 
 max_pooling2d_4 (MaxPooling  (None, 54, 54, 64)       0         
 2D)                                                             
                                                                 
 conv2d_5 (Conv2D)           (None, 52, 52, 128)       73856     
                                                                 
 max_pooling2d_5 (MaxPooling  (None, 26, 26, 128)     

KeyboardInterrupt: 

In [ ]:
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']

epochs_range = range(len(acc))

plt.figure(figsize=(12, 5))

# 정확도 그래프
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Training Accuracy')
plt.plot(epochs_range, val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.title('Training and Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')

# 손실 그래프
plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Training Loss')
plt.plot(epochs_range, val_loss, label='Validation Loss')
plt.legend(loc='upper right')
plt.title('Training and Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')

plt.show()